# 02 — Phishing text classifier

Primary Jupyter workflow for an exploratory, within-dataset phishing text TF-IDF + Logistic Regression experiment with hostname-grouped splits.

Labels remain the original source labels (0 = benign according to source; 1 = phishing according to source). Promotional review flags and their connected components remain quarantined. This is not independent real-world validation.

Run cells in order when ready. Grouping and training may be expensive. This notebook is created without executing its cells. It never opens URLs or makes network/DNS requests.

## Environment and reproducibility

Verify the active kernel and versions. Seed is 20261026. Model configuration matches the existing baseline. The old model and its predictions are not loaded or used for split selection or tuning.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv, json, platform, random, sys

import joblib
import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

SEED = 20261026
random.seed(SEED)
np.random.seed(SEED)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "backend/ml/train_text_baseline.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "backend/ml/train_text_baseline.py").is_file():
    raise RuntimeError("Start this notebook from the repository or one of its directories.")
print({"python": sys.version, "executable": sys.executable,
       "scikit_learn": sklearn.__version__, "numpy": np.__version__,
       "joblib": joblib.__version__, "seed": SEED})

{'python': '3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]', 'executable': 'c:\\Users\\Sushanth CD\\AppData\\Local\\Programs\\Python\\Python314\\python.exe', 'scikit_learn': '1.8.0', 'numpy': '2.4.4', 'joblib': '1.5.3', 'seed': 20261026}


## Load, validate, and protect outputs

Reuse the existing candidate loader, normalization/grouping implementation, and offline hostname extractor. Candidate and raw datasets remain read-only. New outputs have separate paths; preflight refuses to overwrite prior artifacts.

In [2]:
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from backend.ml import train_text_baseline as baseline
from backend.ml.text_normalization import normalize_for_model
from backend.notebooks import create_domain_grouped_split as domain_split

DATA = ROOT / "backend/data/processed/phishing_text_candidates.jsonl"
BASELINE_METRICS = ROOT / "backend/reports/ml/text_phishing_baseline/metrics.json"
MANIFEST = ROOT / "backend/data/processed/domain_grouped_split_manifest.jsonl"
OUT = ROOT / "backend/reports/ml/text_phishing_domain_grouped"
MODEL = ROOT / "backend/models/text/phishing_tfidf_logreg_domain_grouped.joblib"
METRICS, REPORT = OUT / "metrics.json", OUT / "experiment_report.md"
PREDICTIONS, ERRORS, QUARANTINE = (OUT / "test_predictions.csv", OUT / "test_errors.csv", OUT / "quarantine_predictions.csv")
existing = [str(p) for p in (MANIFEST, OUT, MODEL) if p.exists()]
if existing:
    raise FileExistsError("Refusing to overwrite: " + ", ".join(existing))
if not DATA.is_file():
    raise FileNotFoundError(DATA)
if not BASELINE_METRICS.is_file():
    raise FileNotFoundError(BASELINE_METRICS)

records, load_summary = baseline.load_records(DATA)
print(json.dumps(load_summary, indent=2))
print({"records": len(records), "labels": dict(Counter(r["_label"] for r in records)),
       "review_flags": sum(r["review_required"] for r in records),
       "normalized_text_groups": len({r["_normalized_hash"] for r in records})})

{
  "candidate_records": 77676,
  "schema_fields": [
    "content_signals",
    "duplicate_count",
    "duplicate_group_sha256",
    "duplicate_label_conflict",
    "inferred_input_type",
    "label_original",
    "review_reason",
    "review_required",
    "source_dataset",
    "source_record_index",
    "text"
  ],
  "label_counts": {
    "0": 44975,
    "1": 32701
  },
  "review_reason_counts": {
    "possible_promotional_or_non_phishing_spam_review": 696
  },
  "input_type_counts": {
    "html": 15758,
    "url": 41195,
    "plain_text": 18929,
    "email": 1794
  },
  "source_dataset_values": [
    "combined_reduced.json"
  ]
}
{'records': 77676, 'labels': {0: 44975, 1: 32701}, 'review_flags': 696, 'normalized_text_groups': 77411}


## Reuse normalized-text, near-duplicate, and quarantine policy

baseline.create_groups applies the existing NFKC/casefold/whitespace normalization, normalized-text hashes, character 5-grams, 64-bit SHA256 SimHash, Hamming radius 3, Jaccard >= 0.90, and existing conservative quarantine policy. These parameters are not changed.

Join those existing components through eligible exact normalized hostnames from the offline domain parser. Quarantine propagates through the combined graph. Host extraction uses the documented URL/HTML destination rules, exact hostnames (not registrable domains), and no network access.

In [3]:
groups, near_summary = baseline.create_groups(records)
base_members, base_quarantine, hosts_to_components = defaultdict(list), defaultdict(set), defaultdict(set)
destination = {}
for g in groups:
    cid = g["component_id"]
    base_members[cid].append(g)
    base_quarantine[cid].update(g["quarantine_reasons"])
    hosts, methods = set(), set()
    for row in g["records"]:
        found, method = domain_split.destination_hosts(row)
        hosts.update(found)
        methods.add(method)
    destination[g["normalized_hash"]] = {"hosts": sorted(hosts), "method": ";".join(sorted(methods))}
    for host in hosts:
        hosts_to_components[host].add(cid)

dsu = domain_split.DisjointSet(list(base_members))
for linked in hosts_to_components.values():
    linked = sorted(linked)
    for cid in linked[1:]:
        dsu.union(linked[0], cid)
members, reasons = defaultdict(list), defaultdict(set)
for cid, gs in base_members.items():
    root = dsu.find(cid)
    members[root].extend(gs)
    reasons[root].update(base_quarantine[cid])

eligible, quarantined = [], []
for root, gs in members.items():
    combined_id = domain_split.stable_id("text-near-host-v1", [g["normalized_hash"] for g in gs])
    excluded = bool(reasons[root])
    for g in gs:
        g["combined_component_id"] = combined_id
        g["partition"] = "quarantined" if excluded else None
        g["combined_quarantine_reasons"] = sorted(reasons[root])
        (quarantined if excluded else eligible).append(g)

print({"near_duplicate_summary": near_summary, "combined_components": len(members),
       "eligible_groups": len(eligible), "eligible_records": sum(len(g["records"]) for g in eligible),
       "quarantined_groups": len(quarantined),
       "quarantined_records": sum(len(g["records"]) for g in quarantined),
       "distinct_hostnames": len(hosts_to_components),
       "groups_with_hosts": sum(bool(destination[g["normalized_hash"]]["hosts"]) for g in groups),
       "groups_without_hosts": sum(not destination[g["normalized_hash"]]["hosts"] for g in groups)})

{'near_duplicate_summary': {'normalized_text_groups': 77411, 'near_duplicate_pairs_joined': 13302, 'near_duplicate_components': 75532, 'quarantined_components': 645, 'near_duplicate_method': {'normalization': 'Unicode NFKC, casefold, collapsed whitespace', 'shingles': 'unique character 5-grams', 'simhash': 'first 64 bits of SHA-256 per shingle', 'maximum_simhash_hamming_distance': 3, 'minimum_character_shingle_jaccard': 0.9, 'connected_component_id': 'SHA-256 of sorted normalized-text hashes'}}, 'combined_components': 60593, 'eligible_groups': 76036, 'eligible_records': 76286, 'quarantined_groups': 1375, 'quarantined_records': 1390, 'distinct_hostnames': 35991, 'groups_with_hosts': 51365, 'groups_without_hosts': 26046}


## Grouped split and leakage validation

Use deterministic 20-fold StratifiedGroupKFold with seed 20261026: folds 0–2 test, 3–5 validation, and 6–19 train. Whole combined components stay together; exact 70/15/15 proportions and exact class balance are not guaranteed. Quarantined components stay outside all primary partitions.

In [5]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# Each normalized-text group is one sample.
# Its combined component ID determines which partition it belongs to.

split_items = []
split_labels = []
split_group_ids = []

for component_id, component_groups in eligible_components.items():
    for text_group in component_groups:
        split_items.append(text_group)
        split_labels.append(int(text_group["label"]))
        split_group_ids.append(component_id)

X_split = np.zeros(len(split_items))
y_split = np.asarray(split_labels)
groups_split = np.asarray(split_group_ids)

print("Text groups:", len(split_items))
print("Connected components:", len(set(split_group_ids)))

print("\nLabel distribution:")
print({
    0: int(np.sum(y_split == 0)),
    1: int(np.sum(y_split == 1))
})

splitter = StratifiedGroupKFold(
    n_splits=20,
    shuffle=True,
    random_state=20261026
)

folds = list(
    splitter.split(
        X_split,
        y_split,
        groups=groups_split
    )
)

print("\nGenerated folds:", len(folds))

Text groups: 76036
Connected components: 60014

Label distribution:
{0: 44537, 1: 31499}

Generated folds: 20


In [6]:
fold_by_component = {}

for fold_index, (_, heldout_indices) in enumerate(folds):
    for index in heldout_indices:
        component_id = split_group_ids[index]

        existing_fold = fold_by_component.get(component_id)

        if existing_fold is not None and existing_fold != fold_index:
            raise ValueError(
                f"Component assigned to multiple folds: {component_id}"
            )

        fold_by_component[component_id] = fold_index

assert len(fold_by_component) == len(set(split_group_ids))

print("Every component belongs to exactly one fold.")

Every component belongs to exactly one fold.


## Save manifest before fitting

The manifest has one row per source candidate record with stable line identifier, original label, grouping identifiers, partition, hostname status, and quarantine metadata. It contains no predictions or evaluation metrics.

In [7]:
manifest_rows = []
for g in groups:
    info = destination[g["normalized_hash"]]
    for r in g["records"]:
        manifest_rows.append({
            "source_dataset": r["source_dataset"],
            "source_record_id": r.get("source_record_id", "jsonl_line_" + str(r["_line_no"])),
            "source_line": r["_line_no"], "label_original": r["_label"],
            "normalized_text_group_id": g["normalized_hash"],
            "near_duplicate_component_id": g["component_id"],
            "combined_component_id": g["combined_component_id"],
            "partition": g["partition"], "hostnames": info["hosts"],
            "hostname_grouping_status": "grouped" if info["hosts"] else "no_usable_destination_hostname",
            "quarantine_status": "quarantined" if g["partition"] == "quarantined" else "eligible",
            "quarantine_reasons": g["combined_quarantine_reasons"],
            "review_required": r["review_required"], "review_reason": r["review_reason"],
        })
MANIFEST.parent.mkdir(parents=True, exist_ok=True)
with MANIFEST.open("x", encoding="utf-8", newline="\n") as f:
    for item in manifest_rows:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print("Saved manifest rows:", len(manifest_rows), MANIFEST)

Saved manifest rows: 77676 C:\Projects\cyber-security-awareness-platform\backend\data\processed\domain_grouped_split_manifest.jsonl


## TF-IDF + Logistic Regression

Fit on the training partition only. Use the existing baseline configuration and fixed threshold 0.5. Validation is reported but not used for tuning.

In [11]:
print("manifest_rows:", type(manifest_rows), len(manifest_rows))
print("First manifest row:", manifest_rows[0])

print("\nrecords:", type(records), len(records))
print("First record keys:", records[0].keys())

manifest_rows: <class 'list'> 77676
First manifest row: {'source_dataset': 'combined_reduced.json', 'source_record_id': 'jsonl_line_1', 'source_line': 1, 'label_original': 0, 'normalized_text_group_id': '22212838c75a254b7afcbf04617982d8d825d53bc2d5ef4be47f1c5930c78bea', 'near_duplicate_component_id': '0298d28848e58b071c2c0de84579b52fcd111a62ec40af946b3215402aeff554', 'combined_component_id': '5d2dbc326d5cf2e573015f2774e46f5c7e5ef059bbc7c53b1650bb0ca2625db5', 'partition': None, 'hostnames': ['www.clippersharp.com'], 'hostname_grouping_status': 'grouped', 'quarantine_status': 'eligible', 'quarantine_reasons': [], 'review_required': False, 'review_reason': []}

records: <class 'list'> 77676
First record keys: dict_keys(['text', 'label_original', 'source_dataset', 'inferred_input_type', 'review_required', 'review_reason', 'source_record_index', 'duplicate_group_sha256', 'duplicate_count', 'duplicate_label_conflict', 'content_signals', '_line_no', '_label', '_normalized', '_normalized_hash'

In [12]:
from collections import Counter

print("Current partition values:")
print(Counter(row["partition"] for row in manifest_rows))

print("\nRequired variables:")
for name in ["folds", "split_group_ids", "eligible", "quarantined"]:
    print(f"{name}: {name in globals()}")

Current partition values:
Counter({None: 76286, 'quarantined': 1390})

Required variables:
folds: True
split_group_ids: True
eligible: True
quarantined: True


In [13]:
# Map each connected component to its assigned partition.
component_partition = {}

for fold_index, (_, heldout_indices) in enumerate(folds):

    if fold_index < 3:
        partition = "test"
    elif fold_index < 6:
        partition = "validation"
    else:
        partition = "train"

    for index in heldout_indices:
        component_id = str(split_group_ids[index])

        previous = component_partition.get(component_id)

        if previous is not None and previous != partition:
            raise ValueError(
                f"Component crosses partitions: {component_id}"
            )

        component_partition[component_id] = partition


# Identify components excluded by the existing quarantine policy.
quarantined_component_ids = {
    str(group["combined_component_id"])
    for group in quarantined
}

# Add a partition to each source-record manifest entry.
for row in manifest_rows:
    component_id = str(row["combined_component_id"])

    if component_id in component_partition:
        row["partition"] = component_partition[component_id]

    elif component_id in quarantined_component_ids:
        row["partition"] = "quarantined"

    else:
        raise ValueError(
            "Manifest component has no partition or quarantine status: "
            + component_id
        )


print("Partition counts:")
print(Counter(row["partition"] for row in manifest_rows))

Partition counts:
Counter({'train': 53383, 'validation': 11458, 'test': 11445, 'quarantined': 1390})


In [14]:
# Match each manifest entry to its original candidate record.
records_by_line = {
    int(record["_line_no"]): record
    for record in records
}

if len(records_by_line) != len(records):
    raise ValueError("Source record line numbers are not unique.")

manifest_lines = {
    int(row["source_line"])
    for row in manifest_rows
}

if manifest_lines != set(records_by_line):
    raise ValueError(
        "Manifest source lines do not match candidate record lines. "
        "Inspect the record identifiers before proceeding."
    )


def rows_for(partition):
    selected = []

    for manifest_row in manifest_rows:
        if manifest_row["partition"] != partition:
            continue

        source_line = int(manifest_row["source_line"])
        record = records_by_line[source_line]

        # Verify that the original label agrees with the manifest.
        if int(record["_label"]) != int(manifest_row["label_original"]):
            raise ValueError(
                f"Label mismatch at source line {source_line}"
            )

        selected.append(record)

    return selected


train_rows = rows_for("train")
validation_rows = rows_for("validation")
test_rows = rows_for("test")

print("Training records:", len(train_rows))
print("Validation records:", len(validation_rows))
print("Test records:", len(test_rows))
print("Quarantined records:", len(rows_for("quarantined")))

Training records: 53383
Validation records: 11458
Test records: 11445
Quarantined records: 1390


In [15]:
pipeline.fit(
    [r["text"] for r in train_rows],
    [r["_label"] for r in train_rows],
)

print(
    "Training records:", len(train_rows),
    "Vocabulary size:",
    len(pipeline.named_steps["tfidf"].vocabulary_),
)

Training records: 53383 Vocabulary size: 200000


In [16]:
THRESHOLD = 0.5

pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            preprocessor=normalize_for_model,
            lowercase=False,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.995,
            sublinear_tf=True,
            max_features=200000,
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            C=1.0,
            solver="liblinear",
            class_weight=None,
            max_iter=2000,
            random_state=SEED,
        ),
    ),
])

train_rows = rows_for("train")

pipeline.fit(
    [r["text"] for r in train_rows],
    [r["_label"] for r in train_rows],
)

print(
    "Training records:", len(train_rows),
    "Vocabulary size:", len(pipeline.named_steps["tfidf"].vocabulary_),
)

Training records: 53383 Vocabulary size: 200000


## Evaluation and error analysis

Report validation/test metrics, input-type test slices, prediction/error files, and quarantined predictions separately. Quarantine labels remain unadjudicated. Baseline results use a different test partition and potentially different eligible population, so comparisons are contextual and do not establish improvement.

In [17]:
def evaluate(rows):
    y = np.array([r["_label"] for r in rows])
    prob = pipeline.predict_proba([r["text"] for r in rows])[:,1]
    pred = (prob >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    return {"records":len(rows), "tn":int(tn), "fp":int(fp), "fn":int(fn), "tp":int(tp),
            "precision":precision_score(y,pred,zero_division=0),
            "recall":recall_score(y,pred,zero_division=0), "f1":f1_score(y,pred,zero_division=0),
            "specificity":tn/(tn+fp) if tn+fp else None, "accuracy":accuracy_score(y,pred),
            "roc_auc":roc_auc_score(y,prob) if len(set(y)) == 2 else None}
val_rows, test_rows = rows_for("validation"), rows_for("test")
val_metrics, test_metrics = evaluate(val_rows), evaluate(test_rows)
with BASELINE_METRICS.open(encoding="utf-8") as f:
    baseline_payload = json.load(f)
baseline_test_metrics = baseline_payload.get("primary_test_metrics", baseline_payload)
baseline_comparison = {"baseline_test_metrics":baseline_test_metrics, "grouped_test_metrics":test_metrics, "note":"Different test partitions and eligible populations; contextual comparison only."}
print("Baseline comparison (different partitions):", json.dumps(baseline_comparison, indent=2))
print("Validation", json.dumps(val_metrics, indent=2))
print("Test", json.dumps(test_metrics, indent=2))
test_prob = pipeline.predict_proba([r["text"] for r in test_rows])[:,1]
test_pred = (test_prob >= THRESHOLD).astype(int)
pred_rows = [{"source_line":r["_line_no"], "label_original":r["_label"],
              "prediction":int(p), "probability_phishing":float(prob),
              "inferred_input_type":r.get("inferred_input_type"), "text":r["text"]}
             for r,prob,p in zip(test_rows,test_prob,test_pred)]
OUT.mkdir(parents=True, exist_ok=False)
def write_csv(path, rows):
    with path.open("x", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]) if rows else ["empty"])
        writer.writeheader()
        writer.writerows(rows)
write_csv(PREDICTIONS, pred_rows)
write_csv(ERRORS, [r for r in pred_rows if r["prediction"] != r["label_original"]])
slice_metrics = {}
for kind in sorted({str(r.get("inferred_input_type","unknown")) for r in test_rows}):
    slice_metrics[kind] = evaluate([r for r in test_rows if str(r.get("inferred_input_type","unknown")) == kind])
quarantine_rows = [r for g in quarantined for r in g["records"]]
quarantine_metrics = evaluate(quarantine_rows) if quarantine_rows else None
print("Quarantine supplementary metrics (labels unadjudicated):", json.dumps(quarantine_metrics, indent=2))
qprob = pipeline.predict_proba([r["text"] for r in quarantine_rows])[:,1] if quarantine_rows else []
write_csv(QUARANTINE, [{"source_line":r["_line_no"], "label_original":r["_label"],
                        "prediction":int(p>=THRESHOLD), "probability_phishing":float(p),
                        "review_reason":";".join(r["review_reason"]), "text":r["text"]}
                       for r,p in zip(quarantine_rows,qprob)])
print("Input-type slices", json.dumps(slice_metrics, indent=2))
print("Test errors", sum(r["prediction"] != r["label_original"] for r in pred_rows),
      "Quarantined examples scored separately", len(quarantine_rows))

Baseline comparison (different partitions): {
  "baseline_test_metrics": {
    "records": 11496,
    "class_counts": {
      "0": 6738,
      "1": 4758
    },
    "confusion_matrix_labels_0_1": [
      [
        6236,
        502
      ],
      [
        465,
        4293
      ]
    ],
    "TN": 6236,
    "FP": 502,
    "FN": 465,
    "TP": 4293,
    "precision": 0.8953076120959332,
    "recall": 0.9022698612862547,
    "f1": 0.8987752538469591,
    "specificity": 0.9254971801721579,
    "accuracy": 0.915883785664579,
    "roc_auc": 0.9738565944644512,
    "roc_auc_note": null
  },
  "grouped_test_metrics": {
    "records": 11445,
    "tn": 6224,
    "fp": 460,
    "fn": 534,
    "tp": 4227,
    "precision": 0.9018561979944527,
    "recall": 0.8878386893509767,
    "f1": 0.8947925486875529,
    "specificity": 0.931178934769599,
    "accuracy": 0.9131498470948012,
    "roc_auc": 0.9733106965368303
  },
  "note": "Different test partitions and eligible populations; contextual comparison

## Save model and experiment reports

Write only to the new model and report paths. Record versions, configuration, seed, split summary, metrics, and artifact paths. Do not save a raw copy of the dataset.

In [19]:
from collections import Counter, defaultdict

# Expected partition names.
expected_partitions = {
    "train",
    "validation",
    "test",
    "quarantined",
}

# Verify that every manifest entry has a valid partition.
partition_values = Counter(
    row.get("partition") for row in manifest_rows
)

print("Current partition counts:", partition_values)

invalid_rows = [
    row for row in manifest_rows
    if row.get("partition") not in expected_partitions
]

if invalid_rows:
    raise ValueError(
        f"{len(invalid_rows)} manifest rows have missing or invalid "
        "partition assignments. Fix the split assignment before saving."
    )

# Record counts by partition and label.
records_by_partition = Counter()
class_counts = defaultdict(Counter)

for row in manifest_rows:
    partition = row["partition"]
    label = str(row["label_original"])

    records_by_partition[partition] += 1
    class_counts[partition][label] += 1

# Count distinct groups in each partition.
group_fields = [
    "normalized_text_group_id",
    "near_duplicate_component_id",
    "combined_component_id",
]

unique_groups_by_partition = {
    field: defaultdict(set)
    for field in group_fields
}

for row in manifest_rows:
    partition = row["partition"]

    for field in group_fields:
        group_id = row.get(field)

        if group_id is not None:
            unique_groups_by_partition[field][partition].add(group_id)

# Verify that every grouping key stays in one partition.
for field in group_fields:
    group_partitions = defaultdict(set)

    for row in manifest_rows:
        group_id = row.get(field)

        if group_id is not None:
            group_partitions[group_id].add(row["partition"])

    overlapping = [
        group_id
        for group_id, partitions in group_partitions.items()
        if len(partitions) > 1
    ]

    if overlapping:
        raise ValueError(
            f"{field} leakage detected across partitions: "
            f"{len(overlapping)} groups"
        )

# Verify hostname isolation.
hostname_partitions = defaultdict(set)

for row in manifest_rows:
    for hostname in row.get("hostnames", []) or []:
        if hostname:
            hostname_partitions[str(hostname).lower().rstrip(".")].add(
                row["partition"]
            )

overlapping_hosts = [
    hostname
    for hostname, partitions in hostname_partitions.items()
    if len(partitions) > 1
]

if overlapping_hosts:
    raise ValueError(
        f"Hostname isolation failed for {len(overlapping_hosts)} hostnames."
    )

# Construct the summary expected by the metrics payload.
split_summary = {
    "records_by_partition": dict(records_by_partition),

    "class_counts_by_partition": {
        partition: dict(class_counts[partition])
        for partition in sorted(expected_partitions)
    },

    "unique_groups_by_partition": {
        field: {
            partition: len(groups.get(partition, set()))
            for partition in sorted(expected_partitions)
        }
        for field, groups in unique_groups_by_partition.items()
    },

    "group_isolation": {
        "normalized_text": "PASS",
        "near_duplicate_component": "PASS",
        "combined_component": "PASS",
        "exact_hostname": "PASS",
    },

    "seed": SEED,
}

print("\nSplit summary:")
print(split_summary)

Current partition counts: Counter({'train': 53383, 'validation': 11458, 'test': 11445, 'quarantined': 1390})

Split summary:
{'records_by_partition': {'train': 53383, 'quarantined': 1390, 'test': 11445, 'validation': 11458}, 'class_counts_by_partition': {'quarantined': {'1': 991, '0': 399}, 'test': {'0': 6684, '1': 4761}, 'train': {'0': 31202, '1': 22181}, 'validation': {'0': 6690, '1': 4768}}, 'unique_groups_by_partition': {'normalized_text_group_id': {'quarantined': 1375, 'test': 11406, 'train': 53224, 'validation': 11406}, 'near_duplicate_component_id': {'quarantined': 1192, 'test': 11189, 'train': 51977, 'validation': 11174}, 'combined_component_id': {'quarantined': 579, 'test': 8852, 'train': 42145, 'validation': 9017}}, 'group_isolation': {'normalized_text': 'PASS', 'near_duplicate_component': 'PASS', 'combined_component': 'PASS', 'exact_hostname': 'PASS'}, 'seed': 20261026}


In [20]:
required_variables = [
    "pipeline",
    "SEED",
    "THRESHOLD",
    "MANIFEST",
    "MODEL",
    "METRICS",
    "REPORT",
    "PREDICTIONS",
    "ERRORS",
    "test_metrics",
    "val_metrics",
    "baseline_comparison",
    "slice_metrics",
    "quarantine_rows",
    "quarantine_metrics",
    "platform",
    "sklearn",
    "joblib",
    "split_summary",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    print("Variables not yet defined:", missing_variables)
else:
    print("All artifact-saving variables are defined.")

All artifact-saving variables are defined.


In [21]:
print("Model path:", MODEL)
print("Model exists:", MODEL.exists())
print("Metrics path:", METRICS)
print("Metrics exists:", METRICS.exists())
print("Report path:", REPORT)
print("Report exists:", REPORT.exists())

Model path: C:\Projects\cyber-security-awareness-platform\backend\models\text\phishing_tfidf_logreg_domain_grouped.joblib
Model exists: True
Metrics path: C:\Projects\cyber-security-awareness-platform\backend\reports\ml\text_phishing_domain_grouped\metrics.json
Metrics exists: False
Report path: C:\Projects\cyber-security-awareness-platform\backend\reports\ml\text_phishing_domain_grouped\experiment_report.md
Report exists: False
